# LangChain 튜토리얼

In [1]:
!pip install langchain langchain-text-splitters bs4 requests
!pip install faiss-cpu -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# API 설정

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("OPENAI_API_KEY")[:10])

sk-svcacct


# LLM 모델, Embedding 모델 로드

In [3]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-5-mini")    #SDP에 기재된 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-small") #SDP에 기재된 모델

print("LLM, Embeddings 로드 완료!")

LLM, Embeddings 로드 완료!


# Vectorstore 생성

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 더미 청크 (나중에 실제 데이터로 교체)
docs = [
    Document(page_content="국민연금공단 이러닝시스템 사업 예산은 5억 4천만원이다.", metadata={"doc_id": "DOC_001", "page": 1}),
    Document(page_content="콘텐츠 개발 관리 요구사항은 LMS 연동 기능을 포함한다.", metadata={"doc_id": "DOC_001", "page": 3}),
    Document(page_content="기초과학연구원 극저온시스템 사업의 납품기한은 6개월이다.", metadata={"doc_id": "DOC_002", "page": 2}),
]

# FAISS 벡터스토어 생성
vectorstore = FAISS.from_documents(docs, embeddings)
print("벡터스토어 생성 완료!")

/var/folders/l7/25v5xnr126nbxqgsrrj9qv4r0000gp/T/ipykernel_54658/2111819022.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


벡터스토어 생성 완료!


# 검색 테스트

In [5]:
query = "국민연금공단 사업 예산이 얼마야?"
results = vectorstore.similarity_search(query, k=2)

for doc in results:
    print(f"[{doc.metadata['doc_id']}] {doc.page_content}")
    print()

[DOC_001] 국민연금공단 이러닝시스템 사업 예산은 5억 4천만원이다.

[DOC_002] 기초과학연구원 극저온시스템 사업의 납품기한은 6개월이다.



# 답변 LLM에 넘겨서 답변 생성

In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
아래 컨텍스트를 참고해서 질문에 답변해줘.
컨텍스트에 없는 내용은 "문서에서 찾을 수 없습니다"라고 답변해.

컨텍스트:
{context}

질문: {question}
""")

# 컨텍스트 합치기
context = "\n".join([doc.page_content for doc in results])

chain = prompt | llm
response = chain.invoke({"context": context, "question": query})
print(response.content)

국민연금공단 이러닝시스템 사업 예산은 5억 4천만원입니다.
